In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import PolynomialFeatures, OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LinearRegression
from sklearn.metrics import r2_score

# 1. Load the generated CSV dataset
df = pd.read_csv('EV_Battery_Range_Data.csv')

X = df.drop(columns=['Max_Range_Km'])
y = df['Max_Range_Km']

# 2. Split the data into training and testing sets
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# 3. Create a preprocessing engine
# - Apply Degree 2 Polynomial features specifically to Temperature
# - Apply One-Hot encoding to the Drive Mode text column
preprocessor = ColumnTransformer(transformers=[
    ('temp_poly', PolynomialFeatures(degree=2, include_bias=False), ['Ambient_Temperature_C']),
    ('mode_encode', OneHotEncoder(drop='first'), ['Drive_Mode'])  # ✅ fixed here
], remainder='passthrough')  # Leaves 'Battery_Age_Years' alone as a linear term

# 4. Chain the preprocessor and Linear Regression into a smooth pipeline
pipeline = Pipeline([
    ('preprocess', preprocessor),
    ('regressor', LinearRegression())
])

# 5. Train the model
pipeline.fit(X_train, y_train)

# 6. Evaluate accuracy
predictions = pipeline.predict(X_test)
print(f"Polynomial Pipeline Test R² Score: {r2_score(y_test, predictions):.4f}")


In [ ]:
from matplotlib import pyplot as plt

In [ ]:
plt.scatter(df['Ambient_Temperature_C'], df['Max_Range_Km'], color='blue', alpha=0.6)
plt.xlabel('Ambient Temperature (°C)')
plt.ylabel('Max Range (Km)')
plt.title('Temperature vs Battery Range')
plt.show()


In [ ]:
df.columns

In [ ]:
# Scatter plot: Actual vs Predicted
plt.figure(figsize=(8,6))
plt.scatter(y_test, predictions, color='blue', alpha=0.6, edgecolor='k')

# Add a reference line (perfect prediction line)
plt.plot([y_test.min(), y_test.max()],
         [y_test.min(), y_test.max()],
         'r--', lw=2)

plt.xlabel("Actual Max Range (Km)")
plt.ylabel("Predicted Max Range (Km)")
plt.title("Actual vs Predicted Battery Range")
plt.show()


In [ ]:
df

In [ ]:
X

In [ ]:
y

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import PolynomialFeatures, OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LinearRegression
from sklearn.metrics import r2_score

# 1. Load the generated CSV dataset
df = pd.read_csv('EV_Battery_Range_Data.csv')

X = df.drop(columns=['Max_Range_Km'])
y = df['Max_Range_Km']

# 2. Split the data into training and testing sets
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# 3. Create a preprocessing engine
# - Apply Degree 2 Polynomial features specifically to Temperature
# - Apply One-Hot encoding to the Drive Mode text column
preprocessor = ColumnTransformer(transformers=[
    ('temp_poly', PolynomialFeatures(degree=2, include_bias=False), ['Ambient_Temperature_C']),
    ('mode_encode', OneHotEncoder(drop='first'), ['Drive_Mode'])
], remainder='passthrough') # Leaves 'Battery_Age_Years' alone as a linear term

# 4. Chain the preprocessor and Linear Regression into a smooth pipeline
pipeline = Pipeline([
    ('preprocess', preprocessor),
    ('regressor', LinearRegression())
])

# 5. Train the model
pipeline.fit(X_train, y_train)

# 6. Evaluate accuracy
predictions = pipeline.predict(X_test)
print(f"Polynomial Pipeline Test R² Score: {r2_score(y_test, predictions):.4f}")

In [ ]:
preprocessor

In [ ]:
ColumnTransformer

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

# 1. Plotting the EV Battery Dataset
ev_df = pd.read_csv('EV_Battery_Range_Data.csv')
fig, ax = plt.subplots(figsize=(8, 6))
sns.scatterplot(
    data=ev_df, 
    x='Ambient_Temperature_C', 
    y='Max_Range_Km', 
    hue='Drive_Mode', 
    palette='viridis', 
    alpha=0.8,
    ax=ax
)
ax.set_title('Raw EV Data: Ambient Temperature vs Max Driving Range', fontsize=12, fontweight='bold')
ax.set_xlabel('Ambient Temperature (°C)')
ax.set_ylabel('Max Driving Range (km)')
ax.grid(True, linestyle='--', alpha=0.5)
plt.tight_layout()
plt.show()

# 2. Plotting the Banking Dataset
bank_df = pd.read_csv('American Banking Data.xlsx - Fully_Cleaned_Data.csv')
bank_df['Age'] = bank_df['Age'].fillna(bank_df['Age'].median()) # Handle missing row

fig, ax = plt.subplots(figsize=(8, 6))
sns.scatterplot(
    data=bank_df, 
    x='Age', 
    y='Balance', 
    alpha=0.5, 
    color='#3B82F6',
    ax=ax
)
ax.set_title('Raw Banking Data: Customer Age vs Account Balance', fontsize=12, fontweight='bold')
ax.set_xlabel('Age (Years)')
ax.set_ylabel('Account Balance ($)')
ax.grid(True, linestyle='--', alpha=0.5)
plt.tight_layout()
plt.show()

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, PolynomialFeatures
from sklearn.linear_model import LinearRegression
from sklearn.metrics import r2_score, root_mean_squared_error

# ==========================================
# STEP 1: LOAD AND SPLIT DATA
# ==========================================
df = pd.read_csv('EV_Battery_Range_Data.csv')

# Machine learning models can't read text strings ('Eco', 'Sport'). 
# We One-Hot Encode 'Drive_Mode' into binary numbers (1s and 0s)
df_encoded = pd.get_dummies(df, columns=['Drive_Mode'], drop_first=True)

X = df_encoded.drop(columns=['Max_Range_Km'])
y = df_encoded['Max_Range_Km']

# Split data 80% for studying (train), 20% for the exam (test)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# ==========================================
# STEP 2: Z-SCORE STANDARDIZATION
# ==========================================
# We initialize our Z-score normalizer
scaler = StandardScaler()

# List columns that are continuous numbers needing a uniform scale
numeric_cols = ['Ambient_Temperature_C', 'Battery_Age_Years']

X_train_scaled = X_train.copy()
X_test_scaled = X_test.copy()

# Compute the average/std on training set and apply the Z-score calculation
X_train_scaled[numeric_cols] = scaler.fit_transform(X_train[numeric_cols])
X_test_scaled[numeric_cols] = scaler.transform(X_test[numeric_cols])

# ==========================================
# STEP 3: TRAINING THE FIRST MODEL (Straight Line)
# ==========================================
linear_model = LinearRegression()
linear_model.fit(X_train_scaled, y_train)

# Get predictions for the test exam
y_pred_linear = linear_model.predict(X_test_scaled)

# ==========================================
# STEP 4: IDENTIFYING ACCURACY AND ERRORS
# ==========================================
r2_linear = r2_score(y_test, y_pred_linear)
rmse_linear = root_mean_squared_error(y_test, y_pred_linear)

print("=== ROUND 1: STRAIGHT LINE MODEL RESULTS ===")
print(f"R² Score (Accuracy Metric): {r2_linear:.4f}")
print(f"Average Error (RMSE): {rmse_linear:.2f} km\n")

# ==========================================
# STEP 5: HANDLING LOW ACCURACY (Upgrading to Polynomial)
# ==========================================
# Our R² score was low (~0.66). To fix it, we upgrade to a Polynomial curve
poly_transformer = PolynomialFeatures(degree=2, include_bias=False)

X_train_poly = poly_transformer.fit_transform(X_train_scaled)
X_test_poly = poly_transformer.transform(X_test_scaled)

# Train a brand new model on our curved polynomial features
poly_model = LinearRegression()
poly_model.fit(X_train_poly, y_train)

# Test our new curved model on the exam data
y_pred_poly = poly_model.predict(X_test_poly)

r2_poly = r2_score(y_test, y_pred_poly)
rmse_poly = root_mean_squared_error(y_test, y_pred_poly)

print("=== ROUND 2: POLYNOMIAL CURVED MODEL RESULTS ===")
print(f"R² Score (Accuracy Metric): {r2_poly:.4f}  <-- Massive Increase!")
print(f"Average Error (RMSE): {rmse_poly:.2f} km     <-- Massive Error Reduction!")